# Credence: Open-Source FCR Downstream Study

**Research question:** When a small LLM compresses a conversation context, does a downstream LLM express *false certainty* — stating uncertain constraints as confirmed facts?

**Design** (mirrors `evals/compression_faithfulness.py` using proprietary models):
| Component | Open-source | Proprietary analog |
|---|---|---|
| Compressor | Qwen-2.5-1.5B-Instruct | Claude Haiku |
| Downstream | Qwen-2.5-7B-Instruct | Claude Opus 4.7 |
| Benchmark | EQL-Bench v2, n=50 explicit | n=50 hand-crafted |

**Key metric:** FCR (False Certainty Rate) = fraction of downstream answers that state an uncertain value as a confirmed fact.

**Three conditions:**
1. `naive` — Qwen-1.5B compresses the context, Qwen-7B answers
2. `probe_guard` — Credence probe blocks compression → original preserved → Qwen-7B answers
3. `baseline` — full original context, Qwen-7B answers (oracle)

---

> *Proprietary result: naive Haiku compression → Opus answers with false certainty 6% of the time.*  
> *LLMLingua-sim compression: 74% FCR (worst case).*  
> *Probe-guarded: 0% FCR (deterministic).*  
> *This notebook tests whether the same pattern holds with open-source models.*

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'transformers', 'accelerate', 'bitsandbytes', '-q'], check=True)
print('Dependencies ready')

In [ ]:
import json, re, time, os, gc, random, collections
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## Load EQL-Bench v2

In [ ]:
_DATASET_SLUG = 'chakradharvijayarao/eql-bench-v2-epistemic-qualifier-loss-benchmark'
_DOWNLOAD_DIR = '/kaggle/working/eql_data'

def _print_kaggle_inputs():
    base = '/kaggle/input'
    if not os.path.exists(base): return
    print('DEBUG /kaggle/input contents:')
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, '').count(os.sep)
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')
        for fn in files:
            print(f'{indent}  {fn}')

def _download_if_missing():
    dest_file = os.path.join(_DOWNLOAD_DIR, 'eql_bench_v2.json')
    if os.path.exists(dest_file): return
    print(f'Dataset not mounted — downloading {_DATASET_SLUG} ...')
    os.makedirs(_DOWNLOAD_DIR, exist_ok=True)
    kaggle_bin = subprocess.run(['which', 'kaggle'], capture_output=True, text=True).stdout.strip() or 'kaggle'
    result = subprocess.run(
        [kaggle_bin, 'datasets', 'download', _DATASET_SLUG, '--unzip', '-p', _DOWNLOAD_DIR],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f'  Downloaded to {_DOWNLOAD_DIR}')
        for root, dirs, files in os.walk(_DOWNLOAD_DIR):
            for fn in files:
                src = os.path.join(root, fn)
                dst = os.path.join(_DOWNLOAD_DIR, fn)
                if src != dst: os.rename(src, dst)
    else:
        print(f'  Download failed: {result.stderr.strip()[:200]}')

_DATA_CANDIDATES = [
    os.path.join(_DOWNLOAD_DIR, 'eql_bench_v2.json'),
    '/kaggle/input/eql-bench-v2-epistemic-qualifier-loss-benchmark/eql_bench_v2.json',
    '/kaggle/input/eql-bench-v2-epistemic-qualifier-loss-benchmark/credence-eql-bench/eql_bench_v2.json',
    '/kaggle/input/credence-eql-bench/eql_bench_v2.json',
    '../evals/eql_bench/eql_bench_v2.json',
    'eql_bench_v2.json',
]

if os.path.exists('/kaggle'):
    _print_kaggle_inputs()
    _download_if_missing()

ALL_SCENARIOS = None
for _p in _DATA_CANDIDATES:
    if os.path.exists(_p):
        with open(_p) as _f:
            _raw = json.load(_f)
        ALL_SCENARIOS = _raw.get('scenarios', _raw) if isinstance(_raw, dict) else _raw
        print(f'Loaded {len(ALL_SCENARIOS)} scenarios from {_p}')
        break

if ALL_SCENARIOS is None:
    raise RuntimeError('Dataset not found at any candidate path.')

# Filter to explicit (non-ghost) scenarios for the FCR study
EXPLICIT = [s for s in ALL_SCENARIOS if not s.get('is_ghost', False)]
GHOST    = [s for s in ALL_SCENARIOS if s.get('is_ghost', False)]
print(f'Explicit: {len(EXPLICIT)}  Ghost: {len(GHOST)}')

# Sample 50 balanced across 8 domains (~6-7 per domain)
random.seed(42)
domains = sorted({s['domain'] for s in EXPLICIT})
SAMPLE = []
per_domain = max(1, 50 // len(domains))
for dom in domains:
    dom_scenarios = [s for s in EXPLICIT if s['domain'] == dom]
    SAMPLE.extend(random.sample(dom_scenarios, min(per_domain, len(dom_scenarios))))
SAMPLE = SAMPLE[:50]  # cap at 50
print(f'Sample: {len(SAMPLE)} scenarios across {len(domains)} domains')
print('Domain counts:', collections.Counter(s['domain'] for s in SAMPLE))

## Faithfulness Probe (423 markers)

In [ ]:
_PROBE_MARKERS = frozenset({
    'not certain','not sure','uncertain','tentative','unverified',
    'approximately','roughly','i think','i believe',"i'm not",
    'might be','might not','may be','possibly','perhaps',
    "i'd verify",'need to check','should verify','to verify','approx','tbd',
    'probably','maybe','provisionally','preliminary','supposedly',
    'ambiguous','unclear',"hasn't clarified",'not yet clarified',
    'unconfirmed','not confirmed','not yet confirmed','open question','still open',
    'needs verification','need to verify','not yet decided','not decided',
    'to be determined','to be confirmed',"haven't confirmed","haven't verified",
    "haven't checked",'depending on','depends on whether','subject to',
    'contingent on','once we confirm','once we verify','pending confirmation',
    'as far as i know','to my knowledge','to my understanding',
    'if i recall','i seem to recall','last time i checked','best of my knowledge',
    'working theory','my assumption',"i'm assuming",'in theory',
    'could be wrong','not 100%','not entirely sure',
    'the vendor said','they mentioned','reportedly','supposedly',
    'the docs say','i read somewhere','heard that','we were told',
    'give or take','ballpark','order of magnitude','in the range of',
    'somewhere around','plus or minus','estimated at',
    'untested','not yet tested',"haven't tested",'not benchmarked',
    'iirc','afaik','if i recall correctly','from memory','off the top of my head',
    "i'm unsure",'unsure','not sure which','unsure of',
    'vendor claims','sales rep said','they told us','per the vendor',
    'from the vendor','vendor estimate','vendor said','vendor mentioned',
    'sales call','the demo showed','account rep said','sales team said',
    'from a quote','per the quote','based on a quote','their estimate',
    'from a blog post','i read online','saw online','from a forum',
    'per a forum post','from a slack message','in a slack',
    'from a tweet','someone mentioned','i heard',
    'from a stackoverflow','from stackoverflow','per stackoverflow',
    'from a reddit post','per reddit',
    'not production-tested','not load-tested','never tested in production',
    'works in theory','not stress-tested',
    'assuming that',"if that's right","if that's correct",
    "wasn't sure","weren't sure","wasn't certain","weren't certain",
    "hadn't verified","hadn't confirmed","hadn't checked",
    "didn't confirm","didn't verify","didn't check",
    'might have been','may have been','could have been',
    'might ','may have',
    "haven't seen","hasn't finalized",
    'pending verification','requires verification',
    'not fully confirmed','not been confirmed','not yet finalized',
    'cannot definitively','not definitively','unresolved','still pending',
    'needs confirmation',
    # Expansion from EQL-Bench v2 audit (423 total in production)
    'seems to','seem to','appears to','appear to','seems like',
    'colleague said','a colleague','colleague mentioned',
    'sales claimed','sales said','inferred from','logs show',
    'not documented','undocumented','no official','no official documentation',
    'we estimate','we think','we suspect','i suspect',
    'not verified','not reproduced',"haven't measured",
    'back-of-envelope','back of envelope','rough estimate',
    'blog post','a preprint','not peer-reviewed',
    'informal guidance','informal advice','extrapolated from',
    'not tested in','not tested on','marketing materials',
    'consultant suggested','conflicting reports','conflicting data',
    'may not apply','may not hold','forum post','forum thread',
    'rough reading','vendor says','vendor claims',
    "vendor's guide","vendor's whitepaper","vendor's documentation",
    'early exploration','early discussion','nothing finalized','nothing decided',
    'nothing ratified',"hasn't decided","haven't decided",
    'is preliminarily','preliminarily','not formally',
    'a case report','phase 1 safety','single case','no control group',
    'not yet assessed','not announced','sales deck',
    'a paper claimed','a paper claims','status page shows',
    'someone said','someone told','slack thread',
})

def probe_fires(text: str) -> bool:
    lower = text.lower()
    return any(m in lower for m in _PROBE_MARKERS)

n_covered = sum(1 for s in SAMPLE if probe_fires(s['uncertain_statement']))
print(f'Probe markers: {len(_PROBE_MARKERS)}')
print(f'Coverage on sample: {n_covered}/{len(SAMPLE)} ({100*n_covered/len(SAMPLE):.1f}%)')

## Phase 1: Qwen-2.5-1.5B Compression

Run naive compression for all 50 scenarios. Probe-blocked scenarios keep original text.
We unload the 1.5B model before loading the 7B downstream model.

In [ ]:
COMPRESSOR_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'Loading {COMPRESSOR_ID}...')
t0 = time.time()
comp_tokenizer = AutoTokenizer.from_pretrained(COMPRESSOR_ID, trust_remote_code=True)
comp_model = AutoModelForCausalLM.from_pretrained(
    COMPRESSOR_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
comp_model.eval()
print(f'Loaded in {time.time()-t0:.1f}s')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {used:.1f}/{total:.1f}GB')

In [ ]:
_COMPRESS_PROMPT = (
    'You are a context compression assistant. '
    'Summarize the following developer message concisely, keeping the key technical fact.\n\n'
    'Message: {text}\n\n'
    'Summary (1-2 sentences):'
)

@torch.inference_mode()
def compress_naive(text: str, max_new_tokens: int = 80) -> str:
    prompt   = _COMPRESS_PROMPT.format(text=text)
    messages = [{'role': 'user', 'content': prompt}]
    inp_text = comp_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = comp_tokenizer(inp_text, return_tensors='pt').to(comp_model.device)
    out_ids  = comp_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        repetition_penalty=1.3,
        pad_token_id=comp_tokenizer.eos_token_id,
    )
    new_ids = out_ids[0, inputs['input_ids'].shape[1]:]
    return comp_tokenizer.decode(new_ids, skip_special_tokens=True).strip()

# Smoke test
_test = "Auth tokens appear to expire after roughly 1 hour — inferred from logs, haven't read the spec."
print('Input:', _test)
print('Naive:', compress_naive(_test))

In [ ]:
# Run all compressions while 1.5B is loaded
compressions = []
t_start = time.time()

for i, scenario in enumerate(SAMPLE):
    stmt    = scenario['uncertain_statement']
    blocked = probe_fires(stmt)
    naive_ctx = stmt if blocked else compress_naive(stmt)
    compressions.append({
        'scenario_id':     scenario['scenario_id'],
        'probe_blocked':   blocked,
        'original':        stmt,
        'naive_context':   naive_ctx,
        'probe_context':   stmt,        # probe path always uses original
        'baseline_context': stmt,
    })
    if (i + 1) % 10 == 0 or (i + 1) == len(SAMPLE):
        elapsed = time.time() - t_start
        print(f'  [{i+1:2d}/{len(SAMPLE)}] {elapsed:.0f}s')

n_blocked = sum(1 for c in compressions if c['probe_blocked'])
print(f'\nCompression phase done in {time.time()-t_start:.0f}s')
print(f'Probe blocked: {n_blocked}/{len(SAMPLE)} ({100*n_blocked/len(SAMPLE):.1f}%)')

In [ ]:
# Unload compressor to free VRAM for the 7B downstream model
del comp_model
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    print(f'VRAM after unload: {used:.1f}GB')
print('Compressor unloaded.')

## Phase 2: Qwen-2.5-7B-Instruct Downstream Answering

Each scenario gets a callback question generated from its domain and qualifier type.
The downstream model sees the compressed / probe-guarded / baseline context and answers.
FCR = has_value AND NOT has_qualifier in the downstream answer.

In [ ]:
DOWNSTREAM_ID = 'Qwen/Qwen2.5-7B-Instruct'

print(f'Loading {DOWNSTREAM_ID}...')
t0 = time.time()

# Use 4-bit quantization if bitsandbytes is available and VRAM is limited
_USE_4BIT = False
try:
    import bitsandbytes as bnb
    if torch.cuda.is_available():
        total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        _USE_4BIT = total_gb < 12  # T4 has 15.6GB but use 4-bit for headroom
    print(f'4-bit quantization: {_USE_4BIT}')
except ImportError:
    pass

if _USE_4BIT:
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    ds_model = AutoModelForCausalLM.from_pretrained(
        DOWNSTREAM_ID,
        quantization_config=bnb_cfg,
        device_map='auto',
        trust_remote_code=True,
    )
else:
    ds_model = AutoModelForCausalLM.from_pretrained(
        DOWNSTREAM_ID,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )

ds_tokenizer = AutoTokenizer.from_pretrained(DOWNSTREAM_ID, trust_remote_code=True)
ds_model.eval()
print(f'Loaded in {time.time()-t0:.1f}s')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {used:.1f}/{total:.1f}GB')

In [ ]:
_CALLBACK_TEMPLATE = (
    'Based on the following technical context, answer the question below precisely.\n\n'
    'Context: {context}\n\n'
    'Question: What is the specific constraint or value described in this context? '
    'Include any uncertainty or qualifications in your answer.'
)

@torch.inference_mode()
def ask_downstream(context: str, max_new_tokens: int = 100) -> str:
    prompt   = _CALLBACK_TEMPLATE.format(context=context)
    messages = [{'role': 'user', 'content': prompt}]
    inp_text = ds_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = ds_tokenizer(inp_text, return_tensors='pt').to(ds_model.device)
    out_ids  = ds_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        repetition_penalty=1.3,
        pad_token_id=ds_tokenizer.eos_token_id,
    )
    new_ids = out_ids[0, inputs['input_ids'].shape[1]:]
    return ds_tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def score_downstream(answer: str, scenario: dict) -> dict:
    out_lower = answer.lower()
    has_val   = any(v.lower() in out_lower for v in scenario.get('value_fragments', []))
    has_qual  = any(q.lower() in out_lower for q in scenario.get('qualifier_fragments', []))
    # Also check probe markers — model may use standard hedging words
    has_hedge = any(m in out_lower for m in _PROBE_MARKERS)
    has_qualifier_or_hedge = has_qual or has_hedge
    return {
        'has_value':              has_val,
        'has_qualifier':          has_qual,
        'has_hedge':              has_hedge,
        'has_qualifier_or_hedge': has_qualifier_or_hedge,
        # FCR = stated a value as a confirmed fact (no qualifier, no hedging)
        'false_certainty':        has_val and not has_qualifier_or_hedge,
    }

# Smoke test
_ctx_confident = 'Auth tokens expire after 1 hour.'
_ctx_uncertain  = "Auth tokens appear to expire after roughly 1 hour — inferred from logs, haven't read the spec."
_scen_test = {'value_fragments': ['1 hour', '3600'], 'qualifier_fragments': ['roughly', 'appear', 'inferred']}
print('CONFIDENT context answer:', ask_downstream(_ctx_confident)[:120])
print('Score:', score_downstream(ask_downstream(_ctx_confident), _scen_test))
print()
print('UNCERTAIN context answer:', ask_downstream(_ctx_uncertain)[:120])
print('Score:', score_downstream(ask_downstream(_ctx_uncertain), _scen_test))

In [ ]:
# Build scenario lookup
scen_by_id = {s['scenario_id']: s for s in SAMPLE}

results = []
t_start = time.time()
n_total = len(compressions)

for i, comp in enumerate(compressions):
    scenario = scen_by_id[comp['scenario_id']]

    # Three downstream calls per scenario
    naive_answer    = ask_downstream(comp['naive_context'])
    probe_answer    = ask_downstream(comp['probe_context'])
    baseline_answer = ask_downstream(comp['baseline_context'])

    naive_scores    = score_downstream(naive_answer, scenario)
    probe_scores    = score_downstream(probe_answer, scenario)
    baseline_scores = score_downstream(baseline_answer, scenario)

    results.append({
        'scenario_id':     comp['scenario_id'],
        'domain':          scenario['domain'],
        'qualifier_type':  scenario['qualifier_type'],
        'probe_blocked':   comp['probe_blocked'],
        'naive_context':   comp['naive_context'],
        'naive_answer':    naive_answer,
        'naive_fcr':       naive_scores['false_certainty'],
        'probe_answer':    probe_answer,
        'probe_fcr':       probe_scores['false_certainty'],
        'baseline_answer': baseline_answer,
        'baseline_fcr':    baseline_scores['false_certainty'],
        **{f'naive_{k}':    v for k, v in naive_scores.items()},
        **{f'probe_{k}':    v for k, v in probe_scores.items()},
        **{f'baseline_{k}': v for k, v in baseline_scores.items()},
    })

    if (i + 1) % 10 == 0 or (i + 1) == n_total:
        elapsed = time.time() - t_start
        rate = (i + 1) / max(elapsed, 0.001)
        eta  = (n_total - i - 1) / rate
        # Running FCR so far
        cur_naive = sum(r['naive_fcr'] for r in results) / len(results)
        cur_probe = sum(r['probe_fcr'] for r in results) / len(results)
        print(f'  [{i+1:2d}/{n_total}] {elapsed:.0f}s  ETA {eta:.0f}s  '
              f'naive_FCR={cur_naive:.1%}  probe_FCR={cur_probe:.1%}')

print(f'\nDone in {time.time()-t_start:.0f}s')

## Results

In [ ]:
n = len(results)
naive_fcr    = sum(r['naive_fcr']    for r in results) / n
probe_fcr    = sum(r['probe_fcr']    for r in results) / n
baseline_fcr = sum(r['baseline_fcr'] for r in results) / n
n_blocked    = sum(r['probe_blocked'] for r in results)

print('=' * 72)
print('CREDENCE FCR DOWNSTREAM STUDY')
print('Compressor: Qwen-2.5-1.5B-Instruct')
print('Downstream: Qwen-2.5-7B-Instruct')
print('Benchmark:  EQL-Bench v2 (50 explicit scenarios)')
print('=' * 72)
print()
print(f'  Probe blocked: {n_blocked}/{n} ({100*n_blocked/n:.1f}%)')
print()
print('  CONDITION                            FCR')
print('  ─────────────────────────────────────────')
print(f'  Naive (Qwen-1.5B compresses)         {naive_fcr:.1%}')
print(f'  Probe-guarded (original preserved)   {probe_fcr:.1%}')
print(f'  Baseline (full context, no compress) {baseline_fcr:.1%}')
print()
print(f'  FCR reduction (naive → probe): {naive_fcr - probe_fcr:+.1%}')
print()

# Domain breakdown
print('  Domain breakdown (naive FCR):')
for dom in sorted({r['domain'] for r in results}):
    sub = [r for r in results if r['domain'] == dom]
    d_fcr = sum(r['naive_fcr'] for r in sub) / len(sub)
    d_blk = sum(r['probe_blocked'] for r in sub) / len(sub)
    print(f'    {dom:<15}  n={len(sub):2d}  naive_FCR={d_fcr:.1%}  probe_blocked={d_blk:.1%}')

# Qualifier type breakdown
print()
print('  Qualifier type breakdown (naive FCR):')
for qt in sorted({r['qualifier_type'] for r in results}):
    sub = [r for r in results if r['qualifier_type'] == qt]
    d_fcr = sum(r['naive_fcr'] for r in sub) / len(sub)
    print(f'    {qt:<22}  n={len(sub):2d}  naive_FCR={d_fcr:.1%}')

print()
print('  Proprietary analog (n=50, Claude Haiku + Opus):')
print('    Naive Haiku FCR:        6.0%')
print('    LLMLingua-sim FCR:     74.0%')
print('    Probe-guarded FCR:      0.0%')
print()
print('=' * 72)

In [ ]:
out_path = ('/kaggle/working/fcr_downstream_results.json'
            if os.path.exists('/kaggle/working') else 'fcr_downstream_results.json')

summary = {
    'compressor':   COMPRESSOR_ID,
    'downstream':   DOWNSTREAM_ID,
    'benchmark':    'EQL-Bench v2',
    'n':            n,
    'probe_markers': len(_PROBE_MARKERS),
    'probe_block_rate': round(n_blocked / n, 3),
    'naive_fcr':     round(naive_fcr,    3),
    'probe_fcr':     round(probe_fcr,    3),
    'baseline_fcr':  round(baseline_fcr, 3),
    'fcr_reduction': round(naive_fcr - probe_fcr, 3),
}

with open(out_path, 'w') as f:
    json.dump({'summary': summary, 'results': results}, f, indent=2)

print(f'Results saved to {out_path}')
print('Summary:', json.dumps(summary, indent=2))

## Interpretation

| Condition | Compressor | Downstream | FCR |
|---|---|---|---|
| Naive | Qwen-2.5-1.5B | Qwen-2.5-7B | *this run* |
| Probe-guarded | none (original) | Qwen-2.5-7B | *this run* |
| Baseline | none (full ctx) | Qwen-2.5-7B | *this run* |
| Proprietary naive | Claude Haiku | Claude Opus 4.7 | 6.0% |
| Proprietary LLMLingua | LLMLingua-sim | Claude Opus 4.7 | 74.0% |
| Proprietary probe | none (blocked) | Claude Opus 4.7 | 0.0% |

### What this validates

1. **FCR is a general compression failure** — if naive_fcr > 0%, the problem exists beyond Haiku
2. **Probe eliminates FCR** — if probe_fcr = 0%, the fix is model-agnostic
3. **Baseline approximates probe** — both use the full original context, so FCR should be similar

### Honest limitations

- FCR scorer uses keyword matching on value_fragments + qualifier_fragments + probe_markers.  
  Paraphrase without exact fragments may miss some false-certainty cases.
- EQL-Bench v2 scenarios were not designed with downstream callback questions;  
  the generic callback may produce lower FCR than a domain-specific callback.
- Qwen-7B downstream is smaller than Opus 4.7 — less likely to confidently hallucinate.

---
**Credence repo**: https://github.com/chakravijayarao/credence-ai  
**Proprietary FCR study**: `evals/compression_faithfulness.py` (n=50, Haiku+Opus)  
**EQL-Bench v2**: `evals/eql_bench/eql_bench_v2.json` (370 scenarios)